In [1]:
! pip install requests beautifulsoup4 sentence-transformers faiss-cpu google-generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/596.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/596.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/596.4 kB ? eta -:--:--
   --------------------------------- ---- 524.3/596.4 kB 451.7 kB/s eta 0:00:01
   ---------------------------------------- 596.4/596.4 kB 473.2 kB/s  0:00:01
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   -------------------------------

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import random
import re
import os
import faiss
import numpy as np
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

C:\Users\emaha\AppData\Local\Temp\ipykernel_17036\3268238948.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# Scrape Allrecipes Dataset:


HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

CATEGORY_URLS = [
    ("Breakfast",  "https://www.allrecipes.com/recipes/78/breakfast-and-brunch/"),
    ("Dinner",     "https://www.allrecipes.com/recipes/17562/dinner/"),
    ("Desserts",   "https://www.allrecipes.com/recipes/79/desserts/"),
    ("Salads",     "https://www.allrecipes.com/recipes/96/salad/"),
    ("Soups",      "https://www.allrecipes.com/recipes/94/soups-stews-and-chili/"),
]

MAX_RECIPES_PER_CATEGORY = 30   


def get_recipe_links(category_url, max_links=10):
    """Collect recipe page URLs from a category listing page."""
    links = []
    try:
        resp = requests.get(category_url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Allrecipes recipe cards carry href like /recipe/NNNNNN/...
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if re.match(r"https://www\.allrecipes\.com/recipe/\d+/", href):
                if href not in links:
                    links.append(href)
            if len(links) >= max_links:
                break
    except Exception as e:
        print(f"  [link error] {e}")
    return links


def scrape_recipe(url, category, record_id):
    """Scrape a single recipe page and return a structured dict."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        # ── Title ──────────────────────────────────────────────────────────
        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else ""

        # ── Description ───────────────────────────────────────────────────
        desc_tag = soup.find("p", {"class": lambda c: c and "recipe__summary" in " ".join(c)})
        if not desc_tag:
            desc_tag = soup.find("div", {"class": lambda c: c and "recipe__summary" in " ".join(c) if c else False})
        description = desc_tag.get_text(strip=True) if desc_tag else ""

        # ── Ingredients ───────────────────────────────────────────────────
        ingredient_tags = soup.find_all("li", {"class": lambda c: c and "ingredient" in " ".join(c) if c else False})
        if not ingredient_tags:
            ingredient_tags = soup.select("ul.mm-recipes-structured-ingredients__list li")
        ingredients = [i.get_text(" ", strip=True) for i in ingredient_tags if i.get_text(strip=True)]

        # ── Instructions ──────────────────────────────────────────────────
        step_tags = soup.find_all("li", {"class": lambda c: c and "step" in " ".join(c) if c else False})
        if not step_tags:
            step_tags = soup.select("ol.comp.mntl-sc-block-group--OL li")
        instructions = [s.get_text(" ", strip=True) for s in step_tags if s.get_text(strip=True)]

        # Skip if core content missing
        if not title or not ingredients or not instructions:
            return None

        return {
            "record_id": record_id,
            "title": title,
            "source_url": url,
            "category": category,
            "description": description,
            "ingredients": ingredients,
            "instructions": instructions,
        }

    except Exception as e:
        print(f"  [scrape error] {url} → {e}")
        return None


# ── Main scraping loop ──────────────────────────────────────────────────────
all_recipes = []
record_id = 1

for category, cat_url in CATEGORY_URLS:
    print(f"\n Category: {category}")
    links = get_recipe_links(cat_url, MAX_RECIPES_PER_CATEGORY)
    print(f"  Found {len(links)} links")

    for url in links:
        recipe = scrape_recipe(url, category, record_id)
        if recipe:
            all_recipes.append(recipe)
            print(f" [{record_id}] {recipe['title']}")
            record_id += 1
        # Responsible scraping: random delay 2–4 s
        time.sleep(random.uniform(2, 4))

# Save raw data
with open("recipes_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_recipes, f, indent=2, ensure_ascii=False)

print(f"\n Scraped {len(all_recipes)} recipes → recipes_raw.json")


 Category: Breakfast
  Found 8 links
 [1] Oatmeal Breakfast Cookies
 [2] Sheet Pan Brunch Totchos
 [3] Bacon Tater Egg Cups
 [4] Easy Sausage Gravy and Biscuits
 [5] Bauernomlett (Farmer's Omelet)
 [6] Apple Fritter Pancakes
 [7] Leftover Roast Beef Hash
 [8] Pumpkin Waffles

 Category: Dinner
  Found 2 links
 [9] Thai Red Curry Soup
 [10] Campbell's Green Bean Casserole

 Category: Desserts
  Found 2 links
 [11] Watermelon-Mint Paletas
 [12] Orange Party Cake

 Category: Salads
  Found 30 links
 [13] Spinach and Strawberry Salad
 [14] Chef John's Green Goddess Dressing
 [15] Shrimp Pasta Salad
 [16] Sweet and Creamy Cranberry Salad
 [17] Barley, Shrimp, and Corn Salad
 [18] Potato Salad
 [19] Easy Coleslaw Dressing
 [20] Watermelon Salad
 [21] Greek Pasta Salad with Roasted Vegetables and Feta
 [22] Garden Pasta Salad
 [23] No-Mayonnaise Ranch Dressing
 [24] Winter Fruit Salad with Poppyseed Dressing
 [25] Lemon Poppyseed Dressing
 [26] Egg Salad with Olives
 [27] Magical Egg Salad
 

In [5]:

#  Clean Dataset


with open("recipes_raw.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Raw count : {len(raw)}")


def clean_text(text: str) -> str:
    """Strip extra whitespace and non-printable chars."""
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)   # remove control chars
    text = re.sub(r"\s+", " ", text)                  # collapse spaces
    return text.strip()


def clean_recipe(r: dict) -> dict | None:
    """Return a cleaned recipe or None if it should be dropped."""
    title        = clean_text(r.get("title", ""))
    description  = clean_text(r.get("description", ""))
    ingredients  = [clean_text(i) for i in r.get("ingredients", []) if clean_text(i)]
    instructions = [clean_text(s) for s in r.get("instructions", []) if clean_text(s)]

    # Drop if essential fields are empty
    if not title or not ingredients or not instructions:
        return None

    return {
        "record_id"   : r["record_id"],
        "title"       : title,
        "source_url"  : r["source_url"],
        "category"    : r.get("category", ""),
        "description" : description,
        "ingredients" : ingredients,
        "instructions": instructions,
    }


# Clean + deduplicate (by title)
seen_titles = set()
cleaned = []
for r in raw:
    c = clean_recipe(r)
    if c is None:
        continue
    if c["title"].lower() in seen_titles:
        continue
    seen_titles.add(c["title"].lower())
    cleaned.append(c)

with open("recipes_clean.json", "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

print(f"Clean count: {len(cleaned)} → recipes_clean.json")

Raw count : 42
Clean count: 42 → recipes_clean.json


In [6]:

# Chunking Strategy
# Strategy: recipe-level chunking.
# • Ingredients + Instructions stay together in one chunk.
# • Long recipes (>800 chars) are split with 50-char overlap.
# • Every chunk carries full metadata.

MAX_CHUNK_CHARS = 800
OVERLAP_CHARS   = 50


def recipe_to_text(r: dict) -> str:
    """Convert a recipe dict into a single readable string."""
    parts = []
    parts.append(f"Recipe: {r['title']}")
    if r.get("description"):
        parts.append(f"Description: {r['description']}")
    parts.append("Ingredients:")
    for ing in r["ingredients"]:
        parts.append(f"  - {ing}")
    parts.append("Instructions:")
    for i, step in enumerate(r["instructions"], 1):
        parts.append(f"  Step {i}: {step}")
    return "\n".join(parts)


def split_long_text(text: str, max_chars: int, overlap: int) -> list[str]:
    """Split text into overlapping chunks if it exceeds max_chars."""
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap   # overlap with next chunk
    return chunks


with open("recipes_clean.json", "r", encoding="utf-8") as f:
    cleaned = json.load(f)

chunks = []
chunk_id = 0

for recipe in cleaned:
    full_text = recipe_to_text(recipe)
    parts = split_long_text(full_text, MAX_CHUNK_CHARS, OVERLAP_CHARS)

    for part_idx, text_part in enumerate(parts):
        chunks.append({
            "text": text_part,
            "metadata": {
                "chunk_id" : f"chunk_{chunk_id}",
                "title"    : recipe["title"],
                "url"      : recipe["source_url"],
                "category" : recipe["category"],
                "part"     : part_idx,
                "record_id": recipe["record_id"],
            }
        })
        chunk_id += 1

with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"{len(chunks)} chunks → chunks.json")
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=2))

66 chunks → chunks.json

Sample chunk:
{
  "text": "Recipe: Oatmeal Breakfast Cookies\nIngredients:\n  - 3 \u00bc cups old-fashioned oats, divided\n  - \u00bd cup chopped walnuts\n  - \u2153 cup raisins\n  - 2 tablespoons ground flax seeds\n  - 1 teaspoon ground cinnamon\n  - \u00bc teaspoon baking powder\n  - \u00bc teaspoon salt\n  - 2 ripe bananas\n  - 2 large eggs\n  - \u2153 cup unsweetened applesauce\n  - 1 tablespoon honey\n  - 1 tablespoon vanilla extract\nInstructions:\n  Step 1: Preheat the oven to 350 degrees F (175 degrees C). Line 2 baking sheets with parchment paper.\n  Step 2: Grind 2 1/4 cups oats in a food processor to yield 2 cups ground oats. Add walnuts, raisins, flax seeds, cinnamon, baking powder, and salt. Stir in remaining 1 cup oats. Stir thoroughly with a spoon, but do not grind further.\n  Step 3: Make a well in the center of oat mixture. Add bananas, eggs, ap",
  "metadata": {
    "chunk_id": "chunk_0",
    "title": "Oatmeal Breakfast Cookies",
    "url": "h

In [7]:

# Embedding
# Model : all-MiniLM-L6-v2  (sentence-transformers)
# Why   : Fast, lightweight, free, works offline. Good at
#         semantic similarity for short English paragraphs,
#         which matches our recipe chunks perfectly.

with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [c["text"] for c in chunks]

print(" Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f" Embedding {len(texts)} chunks...")
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

# embeddings shape
print(f" Embedding shape: {embeddings.shape}")

# Save embeddings for reuse (avoids re-running if kernel restarts)
np.save("embeddings.npy", embeddings)
print(" Saved → embeddings.npy")

 Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding 66 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

 Embedding shape: (66, 384)
 Saved → embeddings.npy


In [9]:

#  FAISS Indexing

embeddings = np.load("embeddings.npy")
DIM = embeddings.shape[1]          

# FlatL2 = exact nearest-neighbour search using L2 (Euclidean) distance
index = faiss.IndexFlatL2(DIM)
index.add(embeddings.astype(np.float32))

print(f"FAISS index built — {index.ntotal} vectors (dim={DIM})")

# Save index to disk
faiss.write_index(index, "recipe_index.faiss")
print("Saved → recipe_index.faiss")

# Load it back (proves save/load works)
index = faiss.read_index("recipe_index.faiss")
print(f"Loaded index — {index.ntotal} vectors")

FAISS index built — 66 vectors (dim=384)
Saved → recipe_index.faiss
Loaded index — 66 vectors


In [7]:
# Retrieval Function

with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

index = faiss.read_index("recipe_index.faiss")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def retrieve(query: str, top_k: int = 5) -> list[dict]:
    """
    Given a user query, return the top_k most similar recipe chunks.

    Returns a list of dicts, each with:
      - text           : the chunk text
      - metadata       : title, url, category, chunk_id …
      - similarity_score : L2 distance (lower = more similar)
    """
    # 1. Embed the query with the same model used for indexing
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)

    # 2. Search FAISS
    distances, indices = index.search(query_vec, top_k)

    # 3. Build results
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:          # FAISS returns -1 when fewer results exist
            continue
        chunk = chunks[idx]
        results.append({
            "text"            : chunk["text"],
            "metadata"        : chunk["metadata"],
            "similarity_score": float(dist),
        })
    return results


# Quick test
test_results = retrieve("easy chocolate cake recipe")
for r in test_results:
    print(f"[Score: {r['similarity_score']:.3f}] {r['metadata']['title']} — {r['metadata']['url']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Score: 0.979] Orange Party Cake — https://www.allrecipes.com/recipe/15678/orange-party-cake-i/
[Score: 1.114] Oatmeal Breakfast Cookies — https://www.allrecipes.com/recipe/268399/oatmeal-breakfast-cookies/
[Score: 1.167] Oatmeal Breakfast Cookies — https://www.allrecipes.com/recipe/268399/oatmeal-breakfast-cookies/
[Score: 1.211] Easy Sausage Gravy and Biscuits — https://www.allrecipes.com/recipe/216391/easy-sausage-gravy-and-biscuits/
[Score: 1.212] Apple Fritter Pancakes — https://www.allrecipes.com/recipe/8496144/apple-fritter-pancakes/


In [24]:
! pip install -U google-generativeai

In [29]:
! pip install -q google-genai

In [16]:
# Gemini Answer Generation
GEMINI_API_KEY = "YOUR_API_KEY_HERE" 

genai.configure(api_key=GEMINI_API_KEY)

# Use the new flash model
try:
    gemini = genai.GenerativeModel("gemini-2.0-flash-exp")
    print("Using: gemini-2.0-flash-exp")
except Exception as e:
    try:
        gemini = genai.GenerativeModel("gemini-1.5-flash")
        print("Using: gemini-1.5-flash")
    except Exception as e:
        try:
            gemini = genai.GenerativeModel("gemini-pro")
            print("Using: gemini-pro")
        except Exception as e:
            print(f"ERROR: {e}")
            print("Get a new API key from: https://aistudio.google.com/app/apikey")
            exit()

print("Gemini connected")

def build_prompt(query: str, retrieved_chunks: list[dict]):
    context_block = ""
    for i, chunk in enumerate(retrieved_chunks, 1):
        meta = chunk["metadata"]
        context_block += (
            f"--- Source {i} ---\n"
            f"Title: {meta.get('title','')}\n"
            f"Category: {meta.get('category','')}\n"
            f"URL: {meta.get('url','')}\n"
            f"{chunk['text']}\n\n"
        )
    
    prompt = f"""
You are a recipe assistant.

Answer ONLY using the recipe context.

Rules:
- Do not use outside knowledge.
- Do not invent information.
- If missing information say:
"I don't have enough information in the provided recipes."

Context:

{context_block}

Question:

{query}

Answer:
"""
    return prompt

def answer_question(query: str, top_k=5):
    retrieved = retrieve(query, top_k=top_k)
    if not retrieved:
        return "No relevant recipes found."
    prompt = build_prompt(query, retrieved)
    try:
        response = gemini.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {e}"

print("Gemini RAG function ready")

Using: gemini-2.0-flash-exp
Gemini connected
Gemini RAG function ready


In [15]:
# Test Questions
test_questions = [
    "How do I make pancakes for breakfast?",
    "What is a simple chicken soup recipe?",
    "How do I bake a chocolate cake?",
    "Give me an easy salad recipe with tomatoes.",
    "What dessert can I make with eggs and sugar?",
]

for q in test_questions:
    print(f" Question: {q}")
    answer = answer_question(q)
    print(answer)
    print()

 Question: How do I make pancakes for breakfast?
Error: 404 models/gemini-2.0-flash-exp is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.

 Question: What is a simple chicken soup recipe?
Error: 404 models/gemini-2.0-flash-exp is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.

 Question: How do I bake a chocolate cake?
Error: 404 models/gemini-2.0-flash-exp is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.

 Question: Give me an easy salad recipe with tomatoes.
Error: 404 models/gemini-2.0-flash-exp is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of ava